### Realinhamento de features por nome de eletrodo

`degree_i` / `betweenness_i` usavam a ordem local de canais (56–61), então a coluna `i` não era a mesma variável entre sujeitos.

Este notebook:
1. Remapeia `degree_i` → `degree_{Elect}` via `*_nodes.csv`
2. Mantém eletrodos com cobertura ≥95% em EO e EC
3. Grava `data/intermediate_aligned/` e regenera `data/processed/dataset_features.parquet`

Pré-requisito: `01_build_networks.ipynb` (parquets em `data/intermediate/` + CSVs em `networks/`).

Ordem do pipeline de dados: `01_build_networks` → **este notebook** → (opcional revisão em `02_build_dataset`) → EDA / ML.


### Imports e caminhos


In [1]:
from __future__ import annotations

import shutil
from collections import Counter
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
INTERMEDIATE_DIR = PROJECT_ROOT / "data" / "intermediate"
NETWORKS_DIR = INTERMEDIATE_DIR / "networks"
ALIGNED_DIR = PROJECT_ROOT / "data" / "intermediate_aligned"
METADATA_FILE = PROJECT_ROOT / "data" / "metadata" / "participants.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "dataset_features.parquet"
BACKUP_FILE = OUTPUT_DIR / "dataset_features_indexed_backup.parquet"

NODE_METRIC_PREFIXES = (
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
    "sd_degree_",
    "bin_clustering_",
)
COVERAGE_MIN = 0.95

print(PROJECT_ROOT)
print("intermediate exists:", INTERMEDIATE_DIR.exists())


C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC
intermediate exists: True


### Funções de remap e cobertura


In [2]:
def remap_row(features: pd.DataFrame, nodes: pd.DataFrame) -> pd.DataFrame:
    assert len(features) == 1
    elects = nodes["Elect"].astype(str).str.strip().tolist()
    if len(elects) != len(set(elects)):
        raise ValueError("Eletrodos duplicados no CSV de nós.")

    row = features.iloc[0].to_dict()
    new_row: dict = {}
    used_index_cols: set[str] = set()

    for prefix in NODE_METRIC_PREFIXES:
        for i, elect in enumerate(elects):
            old = f"{prefix}{i}"
            if old not in row:
                continue
            new_row[f"{prefix}{elect}"] = row[old]
            used_index_cols.add(old)

    for k, v in row.items():
        if k in used_index_cols or k in ("subject_id", "condition"):
            continue
        skip = False
        for prefix in NODE_METRIC_PREFIXES:
            if k.startswith(prefix) and k[len(prefix):].isdigit():
                skip = True
                break
        if not skip:
            new_row[k] = v

    return pd.DataFrame([new_row])


def coverage_keep(sets: list[set[str]], min_cov: float) -> set[str]:
    counts: Counter[str] = Counter()
    for s in sets:
        counts.update(s)
    n = len(sets)
    return {e for e, k in counts.items() if k / n >= min_cov}


### Remapear intermediários e escolher eletrodos


In [3]:
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)
parquet_files = sorted(INTERMEDIATE_DIR.glob("sub-*.parquet"))
assert parquet_files, f"Nenhum parquet em {INTERMEDIATE_DIR}"

remapped = []
elects_eo, elects_ec = [], []

for path in parquet_files:
    subject_id, condition = path.stem.split("_")
    nodes_path = NETWORKS_DIR / f"{path.stem}_nodes.csv"
    if not nodes_path.exists():
        raise FileNotFoundError(f"Nodes ausente: {nodes_path}")

    feat = pd.read_parquet(path)
    nodes = pd.read_csv(nodes_path)
    aligned = remap_row(feat, nodes)
    aligned["subject_id"] = subject_id
    aligned["condition"] = condition.replace("E0", "EO")
    elect_set = set(nodes["Elect"].astype(str).str.strip())

    remapped.append((subject_id, aligned["condition"].iloc[0], aligned, elect_set))
    if aligned["condition"].iloc[0] == "EO":
        elects_eo.append(elect_set)
    else:
        elects_ec.append(elect_set)

keep = sorted(coverage_keep(elects_eo, COVERAGE_MIN) & coverage_keep(elects_ec, COVERAGE_MIN))
strict_eo = set.intersection(*elects_eo) if elects_eo else set()
strict_ec = set.intersection(*elects_ec) if elects_ec else set()
strict = sorted(strict_eo & strict_ec)

print(
    f"Arquivos: {len(remapped)} | eletrodos @{COVERAGE_MIN:.0%} EO&EC: {len(keep)} | "
    f"intersecao 100% EO&EC: {len(strict)}"
)
print(keep)


Arquivos: 236 | eletrodos @95% EO&EC: 53 | intersecao 100% EO&EC: 25
['AF3', 'AF4', 'AF8', 'AFz', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz', 'Cz', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F8', 'FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'FT7', 'Fp2', 'Fz', 'O1', 'O2', 'Oz', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'PO3', 'PO4', 'PO7', 'PO8', 'PO9', 'POz', 'Pz', 'TP8']


### Gravar `intermediate_aligned`


In [4]:
frames = []
node_cols = [f"{p}{e}" for e in keep for p in NODE_METRIC_PREFIXES]

for subject_id, condition, aligned, _ in remapped:
    out = aligned.copy()
    for col in node_cols:
        if col not in out.columns:
            out[col] = pd.NA
    drop_cols = []
    for c in out.columns:
        for prefix in NODE_METRIC_PREFIXES:
            if c.startswith(prefix):
                elect = c[len(prefix):]
                if elect not in keep:
                    drop_cols.append(c)
                break
    out = out.drop(columns=drop_cols, errors="ignore")
    other = [c for c in out.columns if c not in node_cols]
    out = out[node_cols + other]
    out.to_parquet(ALIGNED_DIR / f"{subject_id}_{condition}.parquet", index=False)
    frames.append(out)

print(f"Salvos {len(frames)} arquivos em {ALIGNED_DIR}")


C:\Users\Anne\AppData\Local\Temp\ipykernel_20624\3754275856.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = pd.NA
C:\Users\Anne\AppData\Local\Temp\ipykernel_20624\3754275856.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = pd.NA
C:\Users\Anne\AppData\Local\Temp\ipykernel_20624\3754275856.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in

Salvos 236 arquivos em C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\intermediate_aligned


### Regenerar `dataset_features.parquet` (filtro saudável + dummies)


In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if OUTPUT_FILE.exists() and not BACKUP_FILE.exists():
    shutil.copy2(OUTPUT_FILE, BACKUP_FILE)
    print(f"Backup: {BACKUP_FILE}")
elif OUTPUT_FILE.exists():
    print(f"Backup já existia: {BACKUP_FILE}")

df_features = pd.concat(frames, ignore_index=True)
meta = pd.read_csv(METADATA_FILE)
meta["subject_id"] = meta["ID"]

df_final = df_features.merge(meta, on="subject_id", how="left", validate="m:1")
df_final = df_final.rename(
    columns={
        "Gender_ 1=female_2=male": "gender",
        "Age": "age",
        "Handedness": "handedness",
        "Education": "education",
        "DRUG_0=negative_1=Positive": "drug",
        "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
        "SKID_Diagnoses": "skid_diagnoses",
        "SKID_Diagnoses 1": "skid_diagnoses_1",
        "SKID_Diagnoses 2": "skid_diagnoses_2",
        "Comments_SKID_assessment": "comments_skid_assessment",
        "Hamilton_Scale": "hamilton_scale",
        "BSL23_sumscore": "bsl23_sumscore",
        "BSL23_behavior": "bsl23_behavior",
        "AUDIT": "audit",
        "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
        "Alcohol_Dependence_In_1st-3rd_Degree_relative": (
            "alcohol_dependence_in_1st_3rd_degree_relative"
        ),
        "Relationship_Status": "relationship_status",
    }
)

df_final["relationship_status"] = df_final["relationship_status"].str.lower()
df_final["handedness"] = df_final["handedness"].str.lower()
df_final["education"] = df_final["education"].str.lower()
df_final["condition"] = df_final["condition"].replace({"E0": "EO"})

healthy_mask = (
    (df_final["drug"] == 0)
    & (df_final["smoking"] == 1)
    & (df_final["audit"] < 8)
    & (df_final["hamilton_scale"] <= 7)
    & (df_final["bsl23_behavior"] == 0)
    & (df_final["skid_diagnoses"].str.lower() == "none")
    & (df_final["skid_diagnoses_2"].isna())
    & (df_final["comments_skid_assessment"].isna())
)
df_healthy = df_final.loc[healthy_mask].copy()
print(
    f"Saudáveis: {df_healthy['subject_id'].nunique()} sujeitos / "
    f"{len(df_healthy)} linhas"
)

meta_cols = ["age", "gender", "handedness", "education", "relationship_status"]
df_meta = pd.get_dummies(df_healthy[meta_cols].copy(), columns=meta_cols, drop_first=True)
df_analysis = pd.concat([df_healthy, df_meta], axis=1)
df_analysis.to_parquet(OUTPUT_FILE, index=False)

named = [
    c for c in df_analysis.columns
    if c.startswith("betweenness_") and not c.split("_", 1)[1].isdigit()
]
indexed = [
    c for c in df_analysis.columns
    if c.startswith("betweenness_") and c.split("_", 1)[1].isdigit()
]
print(f"Salvo: {OUTPUT_FILE} shape={df_analysis.shape}")
print(f"betweenness nomeadas: {len(named)} | indexadas: {len(indexed)}")
assert not indexed, "Ainda há features indexadas"
print("OK")


Backup já existia: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\processed\dataset_features_indexed_backup.parquet


C:\Users\Anne\AppData\Local\Temp\ipykernel_20624\3900019348.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_features = pd.concat(frames, ignore_index=True)


Saudáveis: 54 sujeitos / 108 linhas
Salvo: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\data\processed\dataset_features.parquet shape=(108, 360)
betweenness nomeadas: 53 | indexadas: 0
OK


### Próximos passos

- EDA: `notebook/01_eda/`
- PCA: `notebook/02_pca/`
- Classificação etária: `07` → `08` → `09`

Biblioteca de redes (TVG/REA): `src/graph/` (importada pelos notebooks de construção e pelo app Streamlit).
